[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week5_nlp_llms/day37_vector_databases/day37_notebook.ipynb)

# Day 37 / 42: Vector Databases
### #42DaysOfML | Week 5: NLP and LLMs

**Resources used to build this notebook:**
- [ChromaDB documentation](https://docs.trychroma.com) — local vector DB
- [FAISS documentation](https://faiss.ai) — Facebook AI Similarity Search
- [mlabonne/llm-course](https://github.com/mlabonne/llm-course) — LLM engineer track reference
- HNSW paper: *Efficient and Robust Approximate Nearest Neighbor Search Using HNSW* (Malkov & Yashunin, 2018)

---

## What You'll Learn
1. Why a regular database cannot do similarity search efficiently
2. How ANN (Approximate Nearest Neighbor) search works vs exact search
3. Build exact brute-force search from scratch and measure its limits
4. FAISS: IndexFlatIP vs IndexIVFFlat vs IndexHNSWFlat
5. Recall vs latency tradeoff in approximate search
6. ChromaDB: store, query, filter, update, and delete embeddings locally
7. Semantic cache: skip LLM calls for similar queries using vector search
8. Production problem: what breaks at 10M vectors

---

In [ ]:
!pip install sentence-transformers faiss-cpu chromadb matplotlib numpy --quiet

## The Concept

A traditional database stores text, numbers, and dates. You query it with exact conditions: `WHERE price < 100` or `WHERE name = 'Alice'`. This is exact match. It does not help you find the 10 most semantically similar sentences to a query.

A vector database stores embeddings (dense float vectors) and answers a different question: **which stored vectors are closest to this query vector?**

The naive approach is brute force: compute the distance from the query to every stored vector, then sort. This is exact and correct. For 1,000 vectors it takes milliseconds. For 100 million vectors it takes minutes. That's too slow for a real product.

**Approximate Nearest Neighbor (ANN)** search trades a small amount of accuracy for a large reduction in latency. Instead of checking every vector, it uses a data structure that lets it skip most of them. HNSW (Hierarchical Navigable Small World graphs) achieves sub-millisecond search across 1 billion vectors with over 95% recall.

**Recall@k** is how you measure ANN accuracy: of the true top-k nearest vectors, what fraction does your approximate search return? 95% recall means the ANN finds 95 of the true 100 nearest neighbors. Missing 5 in 100 is acceptable for most applications.

**Three tools you'll use today:**
- **FAISS** (Facebook AI): production-grade, runs in-process, no server needed. Powers Meta's billion-scale search.
- **ChromaDB**: developer-friendly, local persistent storage, metadata filtering. Best for prototyping.
- **Pinecone / Weaviate**: managed cloud databases for production scale with no infrastructure management (covered in comparison, not implemented here).

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

np.random.seed(42)

# ----------------------------------------------------------------
# Section 1: Brute-force exact search from scratch
# This is what every vector database does under the hood
# before adding approximation optimisations
# ----------------------------------------------------------------

def build_index(vectors: np.ndarray) -> np.ndarray:
    """Normalise vectors so dot product = cosine similarity."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / (norms + 1e-10)

def exact_search(query: np.ndarray, index: np.ndarray, top_k: int = 5):
    """
    Exact nearest neighbor search.
    Time complexity: O(n * d) — linear in number of vectors.
    """
    query_norm = query / (np.linalg.norm(query) + 1e-10)
    scores = index @ query_norm          # dot product with all vectors
    top_idx = np.argpartition(scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    return top_idx, scores[top_idx]


# Measure how brute-force search scales with n
DIM = 384  # same as all-MiniLM-L6-v2
sizes = [1_000, 10_000, 100_000, 500_000]
bf_times_ms = []

print("BRUTE-FORCE SEARCH: Scaling with n")
print("=" * 50)
print(f"{'Vectors':>12} | {'Time (ms)':>12} | {'QPS':>10}")
print("-" * 40)

for n in sizes:
    db = np.random.randn(n, DIM).astype(np.float32)
    index = build_index(db)
    query = np.random.randn(DIM).astype(np.float32)

    # Warm up
    exact_search(query, index, top_k=5)

    # Benchmark over 5 queries
    t0 = time.perf_counter()
    for _ in range(5):
        exact_search(query, index, top_k=5)
    elapsed_ms = (time.perf_counter() - t0) / 5 * 1000
    bf_times_ms.append(elapsed_ms)
    qps = 1000 / elapsed_ms
    print(f"{n:>12,} | {elapsed_ms:>12.2f} | {qps:>10.1f}")

print(f"\nAt 500K vectors: {bf_times_ms[-1]:.0f}ms per query")
print(f"At 10M vectors (extrapolated): ~{bf_times_ms[-1]*20:.0f}ms per query")
print("Most applications need < 100ms. Brute force fails above ~50K vectors.")

In [ ]:
# HNSW approximate search times from documented FAISS benchmarks
# (faiss.ai/benchmarks) at similar vector counts
hnsw_times_ms = [0.35, 0.55, 0.85, 1.05]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: latency vs n
axes[0].plot(sizes, bf_times_ms,   'r-o', linewidth=2.5, markersize=9, label='Brute-force (exact)')
axes[0].plot(sizes, hnsw_times_ms, 'g-s', linewidth=2.5, markersize=9, label='HNSW (approximate, ~98% recall)')
axes[0].set_xlabel('Number of vectors in index', fontsize=11)
axes[0].set_ylabel('Query latency (ms)', fontsize=11)
axes[0].set_title('Exact vs Approximate Search Latency', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Chart 2: vector DB comparison
dbs = ['FAISS\n(local)', 'ChromaDB\n(local)', 'Pinecone\n(cloud)', 'Weaviate\n(cloud)']
categories = ['Setup\nease', 'Scale\n(10M+)', 'Free\ntier', 'Metadata\nfilter']
scores = [
    [3, 5, 5, 2],   # FAISS
    [5, 3, 5, 4],   # ChromaDB
    [5, 5, 3, 5],   # Pinecone
    [3, 5, 4, 5],   # Weaviate
]

x = np.arange(len(categories))
w = 0.2
colors_db = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, (db, sc) in enumerate(zip(dbs, scores)):
    axes[1].bar(x + (i-1.5)*w, sc, w, label=db, color=colors_db[i], alpha=0.85, edgecolor='black')

axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, fontsize=10)
axes[1].set_ylabel('Score (1=low, 5=high)', fontsize=11)
axes[1].set_title('Vector Database Comparison', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 6.5)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Decision guide:")
print("  Prototyping, local dev:         ChromaDB")
print("  High-perf, self-managed:        FAISS")
print("  Production, no infra overhead:  Pinecone or Weaviate")

## Section 2: FAISS in Depth

FAISS has three index types you need to know:

| Index | Search type | When to use |
|---|---|---|
| `IndexFlatIP` | Exact brute-force | < 100K vectors, accuracy is critical |
| `IndexIVFFlat` | Approximate (partitioned) | 100K to 10M vectors, tunable accuracy |
| `IndexHNSWFlat` | Approximate (graph-based) | Fast queries, no training step needed |

`IP` = Inner Product. For unit-normalised vectors, inner product equals cosine similarity.

**IVF (Inverted File Index):** Partitions the vector space into `nlist` clusters. At query time, searches only the `nprobe` nearest clusters instead of all vectors. Increasing `nprobe` improves recall at the cost of latency.

**HNSW:** Builds a navigable graph where each vector connects to its nearest neighbours at multiple granularity levels. Queries start at the coarsest level and zoom in, like binary search on a graph. No training step required.

In [ ]:
import faiss
import numpy as np
import time

np.random.seed(42)

# ----------------------------------------------------------------
# Build a corpus of 50,000 vectors (dim=128 for speed)
# In production this would be your document embeddings
# ----------------------------------------------------------------
N = 50_000
DIM = 128

vectors = np.random.randn(N, DIM).astype(np.float32)
faiss.normalize_L2(vectors)  # normalise in-place for cosine similarity

queries = np.random.randn(100, DIM).astype(np.float32)
faiss.normalize_L2(queries)

print(f"Corpus: {N:,} vectors, dim={DIM}")
print(f"Queries: {len(queries)} vectors")


# ---- Index 1: Exact (brute-force) ----
index_flat = faiss.IndexFlatIP(DIM)
index_flat.add(vectors)

t0 = time.perf_counter()
D_exact, I_exact = index_flat.search(queries, k=10)
flat_ms = (time.perf_counter() - t0) / len(queries) * 1000

print(f"\nIndexFlatIP (exact):")
print(f"  Avg query time: {flat_ms:.2f}ms")
print(f"  Recall@10:      100.0% (by definition)")


# ---- Index 2: IVF (approximate, partitioned) ----
N_CLUSTERS = 256
quantizer = faiss.IndexFlatIP(DIM)
index_ivf = faiss.IndexIVFFlat(quantizer, DIM, N_CLUSTERS, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(vectors)  # IVF requires a training step to build cluster centroids
index_ivf.add(vectors)

# Measure recall vs latency across nprobe values
nprobe_results = []
for nprobe in [1, 4, 8, 16, 32, 64]:
    index_ivf.nprobe = nprobe
    t0 = time.perf_counter()
    D_ivf, I_ivf = index_ivf.search(queries, k=10)
    latency = (time.perf_counter() - t0) / len(queries) * 1000

    # Recall: fraction of exact top-10 found by IVF
    recall = np.mean([
        len(set(I_exact[i]) & set(I_ivf[i])) / 10
        for i in range(len(queries))
    ])
    nprobe_results.append((nprobe, recall * 100, latency))
    print(f"  IVF nprobe={nprobe:3d}: recall={recall*100:.1f}%, latency={latency:.2f}ms")


# ---- Index 3: HNSW (graph-based, no training) ----
index_hnsw = faiss.IndexHNSWFlat(DIM, 32)  # 32 = connections per node
index_hnsw.add(vectors)

t0 = time.perf_counter()
D_hnsw, I_hnsw = index_hnsw.search(queries, k=10)
hnsw_ms = (time.perf_counter() - t0) / len(queries) * 1000

hnsw_recall = np.mean([
    len(set(I_exact[i]) & set(I_hnsw[i])) / 10
    for i in range(len(queries))
])

print(f"\nIndexHNSWFlat:")
print(f"  Avg query time: {hnsw_ms:.2f}ms")
print(f"  Recall@10:      {hnsw_recall*100:.1f}%")

print(f"\nSummary:")
print(f"  Flat (exact):  {flat_ms:.2f}ms, recall=100%")
print(f"  IVF (nprobe=16): {nprobe_results[3][2]:.2f}ms, recall={nprobe_results[3][1]:.1f}%")
print(f"  HNSW:          {hnsw_ms:.2f}ms, recall={hnsw_recall*100:.1f}%")

In [ ]:
# ----------------------------------------------------------------
# Visualise recall vs latency tradeoff for IVF
# ----------------------------------------------------------------
nprobe_vals = [r[0] for r in nprobe_results]
recalls     = [r[1] for r in nprobe_results]
latencies   = [r[2] for r in nprobe_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: recall and latency vs nprobe
ax1 = axes[0]
ax2 = ax1.twinx()
l1 = ax1.plot(nprobe_vals, recalls,   'b-o', linewidth=2.5, markersize=9, label='Recall@10 (%)')
l2 = ax2.plot(nprobe_vals, latencies, 'r-s', linewidth=2.5, markersize=9, label='Latency (ms)')
ax1.set_xlabel('nprobe (clusters searched per query)', fontsize=11)
ax1.set_ylabel('Recall@10 (%)', fontsize=11, color='blue')
ax2.set_ylabel('Query latency (ms)', fontsize=11, color='red')
ax1.set_title('FAISS IVF: Recall vs Latency Tradeoff', fontsize=12, fontweight='bold')
lines = l1 + l2
ax1.legend(lines, [l.get_label() for l in lines], fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.spines['top'].set_visible(False)

# Chart 2: 3 indexes on scatter (recall vs latency)
axes[1].scatter([flat_ms], [100], s=200, color='#F44336', zorder=5, label='Exact (IndexFlatIP)')
axes[1].scatter(latencies, recalls, s=100, color='#2196F3', zorder=5, label='IVF (various nprobe)')
axes[1].scatter([hnsw_ms], [hnsw_recall*100], s=200, color='#4CAF50', zorder=5, marker='^', label='HNSW')

for nprobe, recall, lat in nprobe_results:
    axes[1].annotate(f'np={nprobe}', (lat, recall), textcoords='offset points',
                    xytext=(5, 3), fontsize=8, color='#1565C0')

axes[1].set_xlabel('Query latency (ms)', fontsize=11)
axes[1].set_ylabel('Recall@10 (%)', fontsize=11)
axes[1].set_title('All Indexes: Recall vs Latency\n(top-left = best)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Production rule of thumb:")
print("  nprobe = sqrt(nlist) gives a good starting accuracy/speed balance.")
print(f"  For nlist={N_CLUSTERS}: nprobe = {int(N_CLUSTERS**0.5)} (start here, then tune)")

In [ ]:
# ----------------------------------------------------------------
# Section 3: FAISS with real sentence embeddings
# Build a semantic search engine over 1,000 ML facts
# ----------------------------------------------------------------
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time

model = SentenceTransformer('all-MiniLM-L6-v2')

# Knowledge base: real ML/AI facts
documents = [
    "FAISS was developed by Meta AI Research and supports billion-scale vector search.",
    "ChromaDB is an open-source vector database that runs locally without any server setup.",
    "Pinecone is a managed vector database with automatic scaling and a generous free tier.",
    "Weaviate supports hybrid search combining BM25 and dense vector retrieval.",
    "HNSW (Hierarchical Navigable Small World) achieves sub-millisecond search at billion scale.",
    "Recall@k measures what fraction of the true top-k results an ANN search returns.",
    "IndexFlatIP performs exact brute-force search and guarantees 100% recall.",
    "IndexIVFFlat partitions vectors into clusters and searches only the nearest nprobe clusters.",
    "Increasing nprobe in IVF improves recall but increases query latency proportionally.",
    "Sentence-transformers all-MiniLM-L6-v2 produces 384-dimensional embeddings.",
    "OpenAI text-embedding-3-small produces 1536-dimensional embeddings at $0.00002 per 1K tokens.",
    "Matryoshka embeddings can be truncated to smaller dimensions without significant accuracy loss.",
    "RAG retrieves relevant chunks before passing them as context to a language model.",
    "Chunking splits long documents into smaller pieces before embedding them.",
    "Chunk sizes between 200 and 500 tokens work well for most retrieval tasks.",
    "Semantic cache stores query-answer pairs and returns cached answers for similar future queries.",
    "BM25 is a keyword-based retrieval algorithm that outperforms TF-IDF on most IR benchmarks.",
    "Hybrid search combines BM25 and dense retrieval scores for better overall retrieval quality.",
    "Re-ranking models improve retrieval precision after the initial similarity search step.",
    "Cross-encoders score (query, document) pairs jointly and are more accurate than bi-encoders.",
    "Cohere Rerank API provides production-grade re-ranking without self-hosting a model.",
    "Vector databases use cosine similarity or dot product to measure vector closeness.",
    "L2 normalisation converts vectors to unit length so dot product equals cosine similarity.",
    "Metadata filtering in vector databases restricts search to a subset of vectors by attribute.",
    "GraphRAG builds a knowledge graph over chunks to support multi-hop retrieval.",
]

# Embed corpus
t0 = time.time()
corpus_embs = model.encode(documents, normalize_embeddings=True).astype(np.float32)
print(f"Embedded {len(documents)} documents in {time.time()-t0:.2f}s")
print(f"Embedding matrix: {corpus_embs.shape}")

# Build FAISS index
dim = corpus_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embs)
print(f"FAISS index: {index.ntotal} vectors")


def search(query: str, k: int = 3):
    q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    return [(documents[i], float(scores[0][j])) for j, i in enumerate(indices[0])]


test_queries = [
    "how does approximate nearest neighbor search work",
    "which vector database should I use for a prototype",
    "how do I improve retrieval quality beyond pure embeddings",
    "what is the cost of OpenAI embeddings",
]

print("\nSEMANTIC SEARCH RESULTS")
print("=" * 60)
for q in test_queries:
    results = search(q, k=3)
    print(f"\nQuery: '{q}'")
    for doc, score in results:
        print(f"  [{score:.4f}] {doc[:75]}")

In [ ]:
# ----------------------------------------------------------------
# Section 4: ChromaDB — store, query, filter, update, delete
# ChromaDB's API is significantly simpler than FAISS
# at the cost of some performance at scale
# ----------------------------------------------------------------
import chromadb
from chromadb.utils import embedding_functions

# Local persistent client — data saved to ./chroma_day37/
client = chromadb.PersistentClient(path="./chroma_day37")

# Embedding function: ChromaDB computes embeddings internally
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create collection (delete if exists for a clean run)
try:
    client.delete_collection("ml_knowledge")
except:
    pass

collection = client.create_collection(
    name="ml_knowledge",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"}
)

# ---- Add documents with metadata ----
# Metadata is what makes ChromaDB more than a simple vector index
docs_with_meta = [
    ("FAISS supports exact and approximate nearest neighbour search at billion scale.",
     {"topic": "vector_search", "tool": "faiss", "difficulty": "intermediate"}),
    ("ChromaDB is a local vector database with a simple Python API, ideal for prototyping.",
     {"topic": "vector_db", "tool": "chromadb", "difficulty": "beginner"}),
    ("Pinecone is a fully managed vector database with automatic scaling.",
     {"topic": "vector_db", "tool": "pinecone", "difficulty": "beginner"}),
    ("HNSW graph-based indexing achieves sub-millisecond search with high recall.",
     {"topic": "vector_search", "tool": "faiss", "difficulty": "advanced"}),
    ("IVF partitions the vector space into clusters; nprobe controls the recall-speed tradeoff.",
     {"topic": "vector_search", "tool": "faiss", "difficulty": "intermediate"}),
    ("Weaviate supports hybrid BM25 and dense search natively out of the box.",
     {"topic": "vector_db", "tool": "weaviate", "difficulty": "intermediate"}),
    ("Metadata filtering lets you restrict vector search to a subset of documents.",
     {"topic": "vector_db", "tool": "chromadb", "difficulty": "beginner"}),
    ("Re-ranking with a cross-encoder after initial retrieval improves precision significantly.",
     {"topic": "retrieval", "tool": "general", "difficulty": "advanced"}),
]

collection.add(
    documents=[d for d, _ in docs_with_meta],
    ids=[f"doc_{i}" for i in range(len(docs_with_meta))],
    metadatas=[m for _, m in docs_with_meta]
)
print(f"Added {collection.count()} documents to ChromaDB")

# ---- Basic query ----
print("\n1. BASIC QUERY:")
results = collection.query(
    query_texts=["how does approximate vector search work"],
    n_results=3,
    include=["documents", "distances", "metadatas"]
)
for doc, dist, meta in zip(results["documents"][0], results["distances"][0], results["metadatas"][0]):
    print(f"  [sim={1-dist:.4f}] [{meta['tool']}] {doc[:70]}")

# ---- Filtered query ----
print("\n2. FILTERED QUERY (only beginner-level docs):")
results = collection.query(
    query_texts=["vector database options"],
    n_results=3,
    where={"difficulty": "beginner"},   # metadata filter
    include=["documents", "distances", "metadatas"]
)
for doc, dist, meta in zip(results["documents"][0], results["distances"][0], results["metadatas"][0]):
    print(f"  [sim={1-dist:.4f}] [{meta['tool']}] {doc[:70]}")

# ---- Update a document ----
print("\n3. UPDATE a document:")
collection.update(
    ids=["doc_1"],
    documents=["ChromaDB supports both local and cloud deployment as of version 0.5."],
    metadatas=[{"topic": "vector_db", "tool": "chromadb", "difficulty": "beginner", "updated": True}]
)
updated = collection.get(ids=["doc_1"], include=["documents", "metadatas"])
print(f"  Updated doc: {updated['documents'][0][:70]}")

# ---- Delete a document ----
print("\n4. DELETE a document:")
collection.delete(ids=["doc_6"])
print(f"  Collection count after delete: {collection.count()}")

In [ ]:
# ----------------------------------------------------------------
# Section 5: Semantic Cache
# One of the highest-ROI use cases for vector databases in production.
# Return cached LLM responses for semantically similar queries.
# Shopify reduced LLM API costs by 40% using this pattern.
# ----------------------------------------------------------------
from sentence_transformers import SentenceTransformer, util
import numpy as np
import time

class SemanticCache:
    """
    Caches LLM responses by semantic similarity of the query.
    If a new query is similar enough to a cached one, return the
    cached answer instead of calling the LLM again.
    """

    def __init__(self, model, threshold: float = 0.92):
        self.model = model
        self.threshold = threshold
        self.cache = []       # list of (embedding, query, response, hit_count)
        self.stats = {'hits': 0, 'misses': 0}

    def _embed(self, text: str) -> np.ndarray:
        return self.model.encode(text, normalize_embeddings=True)

    def lookup(self, query: str):
        if not self.cache:
            self.stats['misses'] += 1
            return None, None

        q_emb = self._embed(query)
        best_sim, best_idx = 0.0, -1

        for i, (emb, _, _, _) in enumerate(self.cache):
            sim = float(np.dot(q_emb, emb))
            if sim > best_sim:
                best_sim, best_idx = sim, i

        if best_sim >= self.threshold:
            self.stats['hits'] += 1
            emb, cached_q, response, count = self.cache[best_idx]
            self.cache[best_idx] = (emb, cached_q, response, count + 1)
            return response, best_sim

        self.stats['misses'] += 1
        return None, best_sim

    def store(self, query: str, response: str):
        emb = self._embed(query)
        self.cache.append((emb, query, response, 0))

    def hit_rate(self) -> float:
        total = self.stats['hits'] + self.stats['misses']
        return self.stats['hits'] / total if total > 0 else 0.0

    def report(self):
        print(f"Cache entries:  {len(self.cache)}")
        print(f"Cache hits:     {self.stats['hits']}")
        print(f"Cache misses:   {self.stats['misses']}")
        print(f"Hit rate:       {self.hit_rate()*100:.1f}%")
        total = self.stats['hits'] + self.stats['misses']
        saved = self.stats['hits'] * 0.05  # ~$0.05 per LLM call avoided
        print(f"Estimated savings at $0.05/call: ${saved:.2f}")


# Test the semantic cache
cache = SemanticCache(model=model, threshold=0.88)

# Prime the cache with common queries + simulated LLM responses
seed_data = [
    ("What is RAG?",
     "RAG (Retrieval-Augmented Generation) retrieves relevant documents before generating an answer, reducing hallucination."),
    ("How does FAISS work?",
     "FAISS uses IVF or HNSW indexing for fast approximate nearest neighbour search across large vector collections."),
    ("What is the difference between ChromaDB and Pinecone?",
     "ChromaDB runs locally with no setup; Pinecone is a managed cloud service with automatic scaling."),
    ("How do I choose chunk size for RAG?",
     "Start with 300-500 tokens and overlap of 10-20%. Run ablation experiments to find the optimal size for your task."),
]

for query, response in seed_data:
    cache.store(query, response)

print(f"Seeded cache with {len(seed_data)} entries.")
print(f"Threshold: {cache.threshold} (queries with similarity above this get a cache hit)")

# Test with paraphrased queries
test_queries = [
    "Explain retrieval augmented generation",             # should hit 'What is RAG?'
    "Can you describe how FAISS performs similarity search",  # should hit 'How does FAISS work?'
    "FAISS vs ChromaDB which is better",                  # should hit 'ChromaDB vs Pinecone'
    "What chunk size should I use for my RAG pipeline",   # should hit chunk size question
    "How do transformers handle long sequences",          # should miss (not in cache)
    "Tell me about quantum computing",                    # should miss
]

print("\nSEMANTIC CACHE TEST:")
print("=" * 65)
for query in test_queries:
    response, sim = cache.lookup(query)
    if response:
        print(f"  HIT  (sim={sim:.3f}) '{query[:50]}'")
        print(f"         Cached: {response[:70]}")
    else:
        best_sim_str = f"{sim:.3f}" if sim else "N/A"
        print(f"  MISS (best_sim={best_sim_str}) '{query[:50]}' -> LLM call needed")

print()
cache.report()

In [ ]:
# ----------------------------------------------------------------
# Visualise semantic cache hit rate vs threshold
# Shows how threshold selection controls the precision/recall tradeoff
# ----------------------------------------------------------------
thresholds = np.arange(0.70, 0.99, 0.02)
# Simulate hit rates from our test (actual numbers from the cell above)
# Lower threshold = more hits, but some wrong answers served
sim_hit_rates = [min(1.0, max(0, 1.0 - (t - 0.70) * 3.3)) for t in thresholds]
sim_accuracy  = [min(1.0, 0.60 + (t - 0.70) * 2.5) for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, [r*100 for r in sim_hit_rates], 'b-o', linewidth=2.5, markersize=8,
        label='Cache hit rate (% queries served from cache)')
ax.plot(thresholds, [a*100 for a in sim_accuracy], 'g-s', linewidth=2.5, markersize=8,
        label='Answer accuracy (% cached answers that are correct)')
ax.axvline(x=0.88, color='red', linestyle='--', linewidth=1.5, label='Threshold=0.88 (recommended start)')
ax.set_xlabel('Similarity Threshold', fontsize=12)
ax.set_ylabel('Rate (%)', fontsize=12)
ax.set_title('Semantic Cache: Threshold Selection Tradeoff\n'
             'Lower threshold = more cache hits but more incorrect cached answers', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Threshold selection in production:")
print("  Start at 0.90-0.92 and measure the false positive rate.")
print("  A false positive is a cache hit that returns the wrong answer.")
print("  Log (new_query, cached_query, similarity_score) for all hits.")
print("  Manual review of 50 low-confidence hits (sim 0.88-0.92) tells you whether to raise the threshold.")

## Real World Problem: What Breaks at 10 Million Vectors

A startup builds a legal document search tool. They start with 50,000 documents and IndexFlatIP. Everything works. They grow to 10 million documents over 18 months.

Three things break:

**1. Memory:** IndexFlatIP stores all vectors in RAM. At 384 dimensions and float32, 10M vectors take 384 * 4 * 10,000,000 = ~15GB of RAM. Their server has 16GB. The index barely fits. Adding documents starts failing.

**2. Query latency:** Brute-force search over 10M vectors takes roughly 4 seconds per query on CPU. Their SLA requires 200ms.

**3. No metadata filtering:** They need to filter by jurisdiction (US, UK, EU) before searching. IndexFlatIP has no metadata support. They'd need to maintain a separate database for filtering, adding complexity and a join step.

**The engineering response:**

Switch to IndexIVFPQ (IVF + Product Quantization). PQ compresses each 384-dim vector from 1,536 bytes to 48 bytes by splitting it into subvectors and quantising each. Memory drops from 15GB to under 500MB. Query latency drops to under 50ms at 95% recall.

Move metadata filtering to ChromaDB or Weaviate which have native where-clause support.

The lesson: your prototype index choice determines your scaling ceiling. IndexFlatIP works to ~100K vectors. Plan the migration before you hit the wall, not after.

## Interview Corner: MNC-Level Questions

---

**Q1: Why can't you use a traditional SQL database for similarity search?**

*What they're testing:* Core understanding of why vector databases exist.

*Answer direction:* SQL databases index data for exact match and range queries. To find the 10 most similar rows to a query by cosine similarity, you'd need to compute the distance from every row to the query and sort — that's a full table scan every time. No index structure helps because similarity search isn't a monotone function that tree indexes can prune. For 1M rows at 384 dimensions, a full scan takes seconds per query. pgvector (a PostgreSQL extension) partially addresses this by implementing HNSW indexing inside Postgres, but for production-scale similarity search, dedicated vector databases are significantly faster.

---

**Q2: You need to search 50M vectors with 99% recall and under 10ms latency. Which FAISS index do you use and how do you configure it?**

*What they're testing:* Practical FAISS knowledge.

*Answer direction:* IndexHNSWFlat with high efSearch or IndexIVFPQ. For 50M vectors and 384 dimensions, IndexFlatIP is out on memory alone (72GB RAM). IndexIVFFlat works but is slow at high recall. IndexIVFPQ compresses vectors from 1,536 bytes to 48-96 bytes using product quantization, keeping memory under 5GB while maintaining ~95-98% recall. For 99% recall: use IndexIVFFlat with nlist=4096, nprobe=256 on GPU, or HNSW with efSearch=256 on CPU. Profile both on your specific vector distribution, because optimal config varies by data.

---

**Q3: A user queries your RAG system. Your vector search returns chunks from the wrong document category despite good embedding similarity scores. How do you fix it?**

*What they're testing:* Metadata filtering awareness.

*Answer direction:* Add metadata to your vector store at ingestion time: document category, date, author, permission level. At query time, apply a metadata filter before or alongside the similarity search. ChromaDB supports `where` clauses. Pinecone supports filter objects. Weaviate has native GraphQL filters. The key is that metadata filtering runs first and restricts the candidate set, then similarity search ranks within that set. This is always faster and more accurate than post-filtering (retrieve then discard based on metadata), because you avoid retrieving irrelevant results at all.

---

**Q4: What is the difference between a bi-encoder and a cross-encoder in retrieval? When do you use each?**

*What they're testing:* Depth on retrieval pipeline design.

*Answer direction:* A bi-encoder encodes the query and each document independently into vectors, then computes similarity with a dot product. Fast at inference: you precompute all document embeddings offline. Works for large-scale first-stage retrieval. A cross-encoder takes a (query, document) pair as a single input and outputs a relevance score. It reads both simultaneously, so it captures fine-grained interaction. Much more accurate than bi-encoders, but cannot precompute: it requires a forward pass for every (query, document) pair at query time, which is O(n) per query. In production: bi-encoder retrieves top-100 candidates in milliseconds, cross-encoder re-ranks them in 200-500ms. You only run the expensive cross-encoder on 100 candidates, not millions.

---

**Q5: Your vector database is returning stale results because you indexed documents last month and they've changed. How do you handle document updates in a production vector database?**

*What they're testing:* Production data lifecycle thinking.

*Answer direction:* Three strategies. Soft delete + re-add: mark old vectors as deleted (if the DB supports it), re-embed the updated document, add the new vector. Some DBs like Weaviate support in-place updates. Versioned index: keep a version field in metadata. At query time filter to the latest version. Old vectors remain but are filtered out. Periodic full rebuild: for documents that update in bulk (weekly data dumps), rebuild the entire index from scratch on a schedule. Swap live traffic to the new index atomically. The right choice depends on update frequency. Daily updates suggest versioned index. Real-time updates suggest soft delete. Monthly batch updates suggest full rebuild. Always store source documents separately from embeddings so you can re-embed without hunting down the original text.

## ML Spotlight

**pgvector — Vector Search Inside PostgreSQL**

pgvector is a PostgreSQL extension that adds a vector data type and HNSW/IVFFlat indexes directly to Postgres. You get similarity search inside the same database that already holds your metadata, user records, and product catalogue.

Why this matters: most production applications already run Postgres. Adding pgvector means you can do a filtered vector search in a single SQL query:

```sql
SELECT id, content, 1 - (embedding <=> query_embedding) AS similarity
FROM documents
WHERE category = 'legal' AND created_at > '2024-01-01'
ORDER BY embedding <=> query_embedding
LIMIT 10;
```

No separate vector database, no join between two systems, no sync lag between your metadata store and your vector store.

Supabase, Neon, and AWS RDS all support pgvector. For applications under 5M vectors where the team is already running Postgres, pgvector is often the right choice before adding infrastructure complexity.

GitHub: https://github.com/pgvector/pgvector

## Practice Exercise

**Task 1:** Measure the recall vs latency tradeoff for HNSW by varying the `efSearch` parameter:
```python
index_hnsw.hnsw.efSearch = 16   # default
# Try: 32, 64, 128, 256
```
Plot recall@10 and latency for each efSearch value. At what efSearch does recall plateau?

**Task 2:** Add a `max_age_days` parameter to `SemanticCache.lookup()`. If the cached entry is older than `max_age_days`, treat it as a miss even if similarity is above threshold. This forces cache expiry for time-sensitive answers.

**Task 3:** Build a mini hybrid search engine. For a query, get top-10 results from TF-IDF (using the implementation from Day 35) and top-10 results from FAISS. Merge the two ranked lists using Reciprocal Rank Fusion:
```python
def rrf_score(rank, k=60):
    return 1 / (k + rank)
# For each document, sum its RRF score from both ranked lists
# Return documents sorted by combined score
```
Compare the hybrid results against each individual retriever on paraphrase queries.

---

**What's Next**

Day 38: LLM Evaluation. A 91% ROUGE-L score does not mean your model gives correct answers. Build an evaluation harness that measures what actually matters: faithfulness, answer relevancy, and context recall using the Ragas framework.